# Stats Extension: Mixed-Effects Model

This notebook extends the `brain_size.csv` analysis beyond ordinary least squares.

Each person contributes three related IQ scores (`FSIQ`, `VIQ`, `PIQ`). A mixed-effects
model can account for that within-person dependence by giving each subject a
**random intercept**, while still estimating fixed effects for Gender and IQ type.

Tutorial context: https://scipy-lectures.org/packages/statistics/index.html

### Method

1. Reshape IQ scores to long format (one row per subject × IQ type).
2. Fit a linear mixed-effects model:
   - Fixed effects: `Gender`, `iq_type`
   - Random effect: subject intercept
3. Print and interpret the model summary.

In [ ]:
# Import pandas for reading the CSV and reshaping IQ columns to long format.
import pandas as pd

# Import statsmodels formula API so we can fit a mixed linear model with a formula.
# mixedlm is required here because ols cannot include random effects for subjects.
import statsmodels.formula.api as smf

# Load the brain size dataset.
# sep=';' is required because fields are semicolon-separated.
# na_values='.' is required because missing values are coded as '.'.
data = pd.read_csv('brain_size.csv', sep=';', na_values='.')

# Create a unique subject ID from the row index.
# A subject identifier is required so the mixed model knows which IQ scores
# belong to the same person when we reshape to long format.
data['subject'] = data.index

# Reshape FSIQ, VIQ, and PIQ from wide to long format.
# melt is required so each person contributes three rows (one per IQ type),
# which is the structure mixed-effects models expect for repeated measures.
iq_long = data.melt(
    id_vars=['subject', 'Gender', 'MRI_Count', 'Height', 'Weight'],
    value_vars=['FSIQ', 'VIQ', 'PIQ'],
    var_name='iq_type',
    value_name='iq_score',
)

# Quick check of the long-format table used by the model.
print('Long-format IQ data (first 9 rows):')
print(iq_long.head(9))
print()
print('Rows:', len(iq_long), '| Subjects:', iq_long['subject'].nunique())
print()

# Fit a linear mixed-effects model:
#   iq_score ~ Gender + iq_type   (fixed effects)
#   random intercept per subject  (groups=subject)
# groups=iq_long['subject'] is required to model within-subject correlation.
mixed_model = smf.mixedlm(
    'iq_score ~ Gender + iq_type',
    iq_long,
    groups=iq_long['subject'],
)

# Estimate the model parameters.
# .fit() is required to obtain coefficients, variance components, and p-values.
mixed_result = mixed_model.fit()

# Print the full mixed-model summary.
# This reports fixed-effect estimates (Gender, IQ type) and the subject
# random-effect variance (Group Var).
print(mixed_result.summary())

# Print the estimated subject-level variance component explicitly.
# A larger Group Var means more of the IQ variation is shared within people.
print()
print('Subject random-intercept variance (Group Var):')
print(mixed_result.cov_re)

# Brief interpretation of the Gender fixed effect after accounting for
# repeated IQ measures within each subject.
print()
print('Interpretation:')
print(
    'This mixed-effects model extends the earlier OLS gender comparison by '
    'using all three IQ scores per person and a random intercept for subject. '
    'Inspect Gender[T.Male] in the summary: if its p-value is large, there is '
    'still little evidence of a gender difference in IQ after accounting for '
    'within-subject dependence.'
)